In [ ]:
import re
import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.cross_decomposition import PLSRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor, GradientBoostingRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.svm import SVR
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import GroupKFold, cross_val_predict

TRAIN_FILE = "Training_Data.xlsx"
TEST_FILE = "Testing_Data.xlsx"
TRAIN_SHEET = "Training_Data"
TEST_SHEET = "Test_Data"
RANDOM_STATE = 42

In [ ]:
def sanitize_column_name(name):
    if pd.isna(name):
        return "unnamed"

    # Handle Excel time object column
    if not isinstance(name, str):
        name = str(name)

    cleaned = name.strip()
    cleaned = cleaned.replace(",", "")
    cleaned = cleaned.replace("/", "_")
    cleaned = cleaned.replace("(", "_").replace(")", "")
    cleaned = re.sub(r"\s+", "_", cleaned)
    cleaned = re.sub(r"[^0-9A-Za-z_]", "", cleaned)
    cleaned = re.sub(r"_+", "_", cleaned).strip("_")


    return cleaned
    # return replacements.get(cleaned, cleaned)


def clean_columns(df):
    df = df.copy()
    new_cols = []

    for i, col in enumerate(df.columns):
        if i == 0:
            # first column in your Excel is the time column
            new_cols.append("Time")
        else:
            new_cols.append(sanitize_column_name(col))

    df.columns = new_cols
    return df

In [ ]:
train_df = pd.read_excel(TRAIN_FILE, sheet_name=TRAIN_SHEET)
test_df = pd.read_excel(TEST_FILE, sheet_name=TEST_SHEET)

train_df = clean_columns(train_df)
test_df = clean_columns(test_df)

# Sort by X_Value if present
if "X_Value" in train_df.columns:
    train_df = train_df.sort_values("X_Value").reset_index(drop=True)

if "X_Value" in test_df.columns:
    test_df = test_df.sort_values("X_Value").reset_index(drop=True)

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)
print("Columns:", train_df.columns.tolist())

Train shape: (4130, 17)
Test shape: (70, 17)
Columns: ['Time', 'X_Value', 'RM_Fiber', 'RM_Lipid', 'RM_Protein', 'RM_Water', 'Pasting_Temp_C', 'Peak_Viscosity_cp', 'Barrel_Moisture', 'Feed_Rate', 'X_Extruder_RPM', 'SME_kJ_kG', 'Die_Temp', 'Pressure', 'Piece_Density_g_cc', 'SEI', 'Sp_Length_mm_g']


In [ ]:
train_df.head()

,Time,X_Value,RM_Fiber,RM_Lipid,RM_Protein,RM_Water,Pasting_Temp_C,Peak_Viscosity_cp,Barrel_Moisture,Feed_Rate,X_Extruder_RPM,SME_kJ_kG,Die_Temp,Pressure,Piece_Density_g_cc,SEI,Sp_Length_mm_g
0,13:36:01,11593.993138,1.0,0.55,7.0,12.25,83.9,1355.0,0.311933,64.883079,351.614664,435.499820,156.977453,754.686924,0.135174,10.240000,30.968689
1,13:36:02,11594.993196,1.0,0.55,7.0,12.25,83.9,1355.0,0.309224,68.001416,351.614664,415.529130,156.977453,749.077754,0.146876,9.936955,29.370629
2,13:36:03,11595.993253,1.0,0.55,7.0,12.25,83.9,1355.0,0.311731,67.888021,351.614664,420.361595,156.965262,753.711416,0.150185,9.310896,30.654926
3,13:36:04,11596.993310,1.0,0.55,7.0,12.25,83.9,1355.0,0.341724,55.131194,351.614664,523.574476,156.965262,750.053262,0.148038,9.902281,29.242054
4,13:36:05,11597.993367,1.0,0.55,7.0,12.25,83.9,1355.0,0.337590,55.017800,351.633603,499.334143,156.965262,742.736953,0.141134,8.879306,34.206219


In [ ]:
TARGET_COLS = ["Piece_Density_g_cc", "SEI", "Sp_Length_mm_g"]

def assign_segments_and_windows(df, window_sec=60, gap_threshold_sec=5.0):
    out = df.copy()

    d = out["X_Value"].diff()
    out["segment_id"] = (d > gap_threshold_sec).cumsum().astype(int)

    seg_start = out.groupby("segment_id")["X_Value"].transform("min")
    out["bag_id"] = np.floor((out["X_Value"] - seg_start) / window_sec).astype(int)
    out["bag_id"] = out["bag_id"].clip(lower=0)

    return out


def build_window_dataset(df, window_sec=60, gap_threshold_sec=5.0):
    second_df = assign_segments_and_windows(df, window_sec, gap_threshold_sec)

    numeric_cols = second_df.select_dtypes(include=[np.number]).columns.tolist()
    protected_cols = {"segment_id", "bag_id", "X_Value"}

    process_cols = [c for c in numeric_cols if c not in protected_cols and c not in TARGET_COLS]
    group_keys = ["segment_id", "bag_id"]

    agg_dict = {c: ["mean", "std", "min", "max"] for c in process_cols}

    for t in TARGET_COLS:
        if t in second_df.columns:
            agg_dict[t] = ["mean", "std", "count"]

    agg = second_df.groupby(group_keys).agg(agg_dict)
    agg.columns = [f"{a}_{b}" for a, b in agg.columns]
    agg = agg.reset_index()

    for c in process_cols:
        min_col, max_col = f"{c}_min", f"{c}_max"
        if min_col in agg.columns and max_col in agg.columns:
            agg[f"{c}_range"] = agg[max_col] - agg[min_col]

    return agg

In [ ]:
train_window_df = build_window_dataset(train_df, window_sec=60, gap_threshold_sec=5.0)
test_window_df = build_window_dataset(test_df, window_sec=60, gap_threshold_sec=5.0)

print("Aggregated train shape:", train_window_df.shape)
print("Aggregated test shape:", test_window_df.shape)

Aggregated train shape: (70, 71)
Aggregated test shape: (70, 71)


In [ ]:
train_window_df.columns

Index(['segment_id', 'bag_id', 'RM_Fiber_mean', 'RM_Fiber_std', 'RM_Fiber_min',
       'RM_Fiber_max', 'RM_Lipid_mean', 'RM_Lipid_std', 'RM_Lipid_min',
       'RM_Lipid_max', 'RM_Protein_mean', 'RM_Protein_std', 'RM_Protein_min',
       'RM_Protein_max', 'RM_Water_mean', 'RM_Water_std', 'RM_Water_min',
       'RM_Water_max', 'Pasting_Temp_C_mean', 'Pasting_Temp_C_std',
       'Pasting_Temp_C_min', 'Pasting_Temp_C_max', 'Peak_Viscosity_cp_mean',
       'Peak_Viscosity_cp_std', 'Peak_Viscosity_cp_min',
       'Peak_Viscosity_cp_max', 'Barrel_Moisture_mean', 'Barrel_Moisture_std',
       'Barrel_Moisture_min', 'Barrel_Moisture_max', 'Feed_Rate_mean',
       'Feed_Rate_std', 'Feed_Rate_min', 'Feed_Rate_max',
       'X_Extruder_RPM_mean', 'X_Extruder_RPM_std', 'X_Extruder_RPM_min',
       'X_Extruder_RPM_max', 'SME_kJ_kG_mean', 'SME_kJ_kG_std',
       'SME_kJ_kG_min', 'SME_kJ_kG_max', 'Die_Temp_mean', 'Die_Temp_std',
       'Die_Temp_min', 'Die_Temp_max', 'Pressure_mean', 'Pressure_std',
  

In [ ]:
def engineer_features_rich(df):
    out = df.copy()

    eps = 1e-8

    # -----------------------------
    # 1. Base aggregated features
    # -----------------------------
    # Keep almost all numeric aggregated columns except IDs and target summaries
    exclude_cols = {
        "segment_id",
        "bag_id",
        "Piece_Density_g_cc_mean",
        "Piece_Density_g_cc_std",
        "Piece_Density_g_cc_count",
        "SEI_mean",
        "SEI_std",
        "SEI_count",
        "Sp_Length_mm_g_mean",
        "Sp_Length_mm_g_std",
        "Sp_Length_mm_g_count",
    }

    numeric_cols = out.select_dtypes(include=[np.number]).columns.tolist()
    rich_features = [c for c in numeric_cols if c not in exclude_cols]

    # -----------------------------
    # 2. Domain interaction terms
    # -----------------------------
    def has(*cols):
        return all(c in out.columns for c in cols)

    if has("X_Extruder_RPM_mean", "Barrel_Moisture_mean"):
        out["RPM_x_Moisture"] = out["X_Extruder_RPM_mean"] * out["Barrel_Moisture_mean"]

    if has("Feed_Rate_mean", "Barrel_Moisture_mean"):
        out["Feed_x_Moisture"] = out["Feed_Rate_mean"] * out["Barrel_Moisture_mean"]

    if has("X_Extruder_RPM_mean", "Feed_Rate_mean"):
        out["RPM_x_Feed"] = out["X_Extruder_RPM_mean"] * out["Feed_Rate_mean"]

    if has("SME_kJ_kG_mean", "Barrel_Moisture_mean"):
        out["SME_x_Moisture"] = out["SME_kJ_kG_mean"] * out["Barrel_Moisture_mean"]

    if has("Die_Temp_mean", "Pressure_mean"):
        out["DieTemp_x_Pressure"] = out["Die_Temp_mean"] * out["Pressure_mean"]

    if has("RM_Fiber_mean", "SME_kJ_kG_mean"):
        out["Fiber_x_SME"] = out["RM_Fiber_mean"] * out["SME_kJ_kG_mean"]

    if has("RM_Protein_mean", "Die_Temp_mean"):
        out["Protein_x_DieTemp"] = out["RM_Protein_mean"] * out["Die_Temp_mean"]

    # -----------------------------
    # 3. Ratio features
    # -----------------------------
    if has("X_Extruder_RPM_mean", "Feed_Rate_mean"):
        out["RPM_over_Feed"] = out["X_Extruder_RPM_mean"] / (out["Feed_Rate_mean"] + eps)

    if has("SME_kJ_kG_mean", "Feed_Rate_mean"):
        out["SME_over_Feed"] = out["SME_kJ_kG_mean"] / (out["Feed_Rate_mean"] + eps)

    if has("Pressure_mean", "Die_Temp_mean"):
        out["Pressure_over_DieTemp"] = out["Pressure_mean"] / (out["Die_Temp_mean"] + eps)

    # -----------------------------
    # 4. Stability / variability features
    # -----------------------------
    candidate_bases = [
        "Feed_Rate",
        "X_Extruder_RPM",
        "SME_kJ_kG",
        "Die_Temp",
        "Pressure",
        "Barrel_Moisture",
        "Pasting_Temp_C",
        "Peak_Viscosity_cp",
    ]

    for base in candidate_bases:
        mean_col = f"{base}_mean"
        std_col = f"{base}_std"
        min_col = f"{base}_min"
        max_col = f"{base}_max"
        range_col = f"{base}_range"

        if has(std_col, mean_col):
            out[f"{base}_cv"] = out[std_col] / (out[mean_col].abs() + eps)

        if has(max_col, min_col):
            out[f"{base}_span"] = out[max_col] - out[min_col]

        if has(range_col, mean_col):
            out[f"{base}_range_over_mean"] = out[range_col] / (out[mean_col].abs() + eps)

    # -----------------------------
    # 5. Formulation ratios
    # -----------------------------
    if has("RM_Protein_mean", "RM_Water_mean"):
        out["Protein_over_Water"] = out["RM_Protein_mean"] / (out["RM_Water_mean"] + eps)

    if has("RM_Fiber_mean", "RM_Water_mean"):
        out["Fiber_over_Water"] = out["RM_Fiber_mean"] / (out["RM_Water_mean"] + eps)

    if has("RM_Lipid_mean", "RM_Protein_mean"):
        out["Lipid_over_Protein"] = out["RM_Lipid_mean"] / (out["RM_Protein_mean"] + eps)

    # -----------------------------
    # 6. Add engineered columns to feature list
    # -----------------------------
    engineered_cols = [
        "RPM_x_Moisture",
        "Feed_x_Moisture",
        "RPM_x_Feed",
        "SME_x_Moisture",
        "DieTemp_x_Pressure",
        "Fiber_x_SME",
        "Protein_x_DieTemp",
        "RPM_over_Feed",
        "SME_over_Feed",
        "Pressure_over_DieTemp",
        "Protein_over_Water",
        "Fiber_over_Water",
        "Lipid_over_Protein",
    ]

    # add generated stability columns if present
    for base in candidate_bases:
        for suffix in ["cv", "span", "range_over_mean"]:
            col = f"{base}_{suffix}"
            if col in out.columns:
                engineered_cols.append(col)

    engineered_cols = [c for c in engineered_cols if c in out.columns]

    # final feature set
    final_features = rich_features + engineered_cols

    # remove duplicates while keeping order
    final_features = list(dict.fromkeys(final_features))

    return out, final_features

train_feat_df, rich_features = engineer_features_rich(train_window_df)
test_feat_df, _ = engineer_features_rich(test_window_df)

In [ ]:
PRIMARY_TARGET = "Piece_Density_g_cc_mean"

train_model_df = train_feat_df.dropna(subset=[PRIMARY_TARGET]).copy()
test_model_df = test_feat_df.dropna(subset=[PRIMARY_TARGET]).copy()

# X_train = train_model_df[core_features].copy()
X_train = train_model_df[rich_features].copy()
y_train = train_model_df[PRIMARY_TARGET].copy()
groups = train_model_df["segment_id"].values

# X_test = test_model_df[core_features].copy()
X_test = test_model_df[rich_features].copy()
y_test = test_model_df[PRIMARY_TARGET].copy()

print("Target being predicted:", PRIMARY_TARGET)

Target being predicted: Piece_Density_g_cc_mean


In [ ]:
def compute_metrics(y_true, y_pred):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    return {"RMSE": rmse, "MAE": mae, "R2": r2}


def make_preprocessor(numeric_features):
    num_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ])

    pre = ColumnTransformer([
        ("num", num_pipe, numeric_features),
    ], remainder="drop")

    return pre


def build_model_registry(random_state=42):
    models = {
        "LinearRegression": LinearRegression(),
        "Ridge": Ridge(alpha=1.0, random_state=random_state),
        "Lasso": Lasso(alpha=1e-3, random_state=random_state, max_iter=5000),
        "ElasticNet": ElasticNet(alpha=1e-3, l1_ratio=0.5, random_state=random_state, max_iter=5000),
        "PLS": PLSRegression(n_components=2),
        "DecisionTree": DecisionTreeRegressor(max_depth=5, random_state=random_state),
        "RandomForest": RandomForestRegressor(
            n_estimators=400, min_samples_leaf=2, random_state=random_state, n_jobs=1
        ),
        "ExtraTrees": ExtraTreesRegressor(
            n_estimators=500, min_samples_leaf=2, random_state=random_state, n_jobs=1
        ),
        "GradientBoosting": GradientBoostingRegressor(random_state=random_state),
        "SVR_RBF": SVR(kernel="rbf", C=10, epsilon=0.01),
        "MLP": MLPRegressor(
            hidden_layer_sizes=(64, 32),
            activation="relu",
            early_stopping=True,
            max_iter=800,
            random_state=random_state,
        ),
    }
    return models


def make_model_pipeline(model, feature_cols):
    pre = make_preprocessor(feature_cols)
    pipe = Pipeline([
        ("pre", pre),
        ("model", model),
    ])
    return pipe

In [ ]:
def evaluate_models(X_train, y_train, X_test, y_test, models, groups, n_splits=5):
    rows = []
    fitted_models = {}

    n_group = len(np.unique(groups))
    n_splits = min(n_splits, n_group) if n_group >= 2 else 2
    cv = GroupKFold(n_splits=n_splits)

    for name, model in models.items():
        pipe = make_model_pipeline(clone(model), X_train.columns.tolist())

        # group-aware CV prediction on training set
        oof_pred = cross_val_predict(pipe, X_train, y_train, cv=cv, groups=groups, n_jobs=1)
        cv_metrics = compute_metrics(y_train.values, oof_pred)

        # fit full training set
        pipe.fit(X_train, y_train)

        train_pred = pipe.predict(X_train)
        test_pred = pipe.predict(X_test)

        train_metrics = compute_metrics(y_train.values, train_pred)
        test_metrics = compute_metrics(y_test.values, test_pred)

        rows.append({
            "Model": name,
            # "Train_R2": train_metrics["R2"],
            "CV_R2": cv_metrics["R2"],
            "Test_R2": test_metrics["R2"],
            # "Train_RMSE": train_metrics["RMSE"],
            "CV_RMSE": cv_metrics["RMSE"],
            "Test_RMSE": test_metrics["RMSE"],
            "Test_MAE": test_metrics["MAE"],
        })

        fitted_models[name] = {
            "pipeline": pipe,
            "pred_test": test_pred,
            "pred_train": train_pred,
        }

    results = pd.DataFrame(rows).sort_values("Test_RMSE").reset_index(drop=True)
    return results, fitted_models


MODEL_REGISTRY = build_model_registry(RANDOM_STATE)

results_df, fitted_models = evaluate_models(
    X_train=X_train,
    y_train=y_train,
    X_test=X_test,
    y_test=y_test,
    models=MODEL_REGISTRY,
    groups=groups,
)

print(results_df)

               Model         CV_R2     Test_R2   CV_RMSE  Test_RMSE  Test_MAE
0         ExtraTrees      0.146664    0.394618  0.020311   0.020313  0.015161
1              Lasso   -452.339661    0.343621  0.468149   0.021152  0.016197
2                PLS    -38.506767    0.296959  0.138200   0.021891  0.016592
3         ElasticNet   -457.989104    0.295881  0.471057   0.021907  0.016886
4       RandomForest      0.305734    0.274172  0.018320   0.022242  0.017361
5            SVR_RBF     -0.133324    0.210204  0.023407   0.023202  0.018218
6   GradientBoosting      0.168524    0.085226  0.020049   0.024970  0.019982
7       DecisionTree     -1.349526   -0.171367  0.033703   0.028256  0.022258
8              Ridge   -172.226052   -0.225620  0.289387   0.028903  0.024207
9   LinearRegression  -2889.361974  -19.411534  1.182084   0.117952  0.100070
10               MLP -30864.537577 -386.151849  3.862861   0.513696  0.459567


In [14]:
best_model_name = results_df.iloc[0]["Model"]
best_model = fitted_models[best_model_name]["pipeline"]

print("Best model:", best_model_name)

test_predictions = pd.DataFrame({
    "actual_piece_density": y_test.values,
    "predicted_piece_density": fitted_models[best_model_name]["pred_test"]
})

print(test_predictions.head())

Best model: ExtraTrees
   actual_piece_density  predicted_piece_density
0              0.151995                 0.152807
1              0.140868                 0.146775
2              0.162678                 0.150297
3              0.215877                 0.152845
4              0.157786                 0.168008
